# 텍스트 생성 및 프롬프트
OpenAI API를 사용하면 ChatGPT처럼 대규모 언어 모델을 사용하여 프롬프트에서 텍스트를 생성할 수 있습니다.  
모델은 코드, 수학 방정식, 구조화된 JSON 데이터 또는 인간과 유사한 산문 등 거의 모든 종류의 텍스트 응답을 생성할 수 있습니다.

## 간단한 프롬프트에서 텍스트 생성

In [1]:
# pip install -U openai

In [2]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

- 메시지는 메시지 개체의 배열이어야 하며, 각 개체에는role(“developer", “user" 또는 “assistant")과 content가 있습니다. 대화는 하나의 메시지만큼 짧을 수도 있고 여러 번 주고받을 수도 있습니다.
- 대화는 먼저 developer 메시지로 형식화되고 이어서 user 메시지와 assistanc 메시지가 교대로 표시됩니다.  
- developer 메시지는 assistant의 동작을 설정하는 데 도움이 됩니다. 예를 들어, assitant의 성격을 수정하거나 대화 전반에 걸쳐 assistant가 어떻게 행동해야 하는지에 대한 구체적인 지침을 제공할 수 있습니다. 그러나 developer 메시지는 선택 사항이며 developer 메시지가 없는 모델의 동작은 "당신은 도움이 되는 조수입니다"와 같은 일반적인 메시지를 사용하는 것과 유사할 가능성이 높습니다.  
- User 메시지는 assistant가 응답할 요청이나 설명을 제공합니다. Assistant 메시지는 이전 assistant 응답을 저장하지만 원하는 동작의 예를 제공하기 위해 user가 작성할 수도 있습니다.

In [11]:
from openai import OpenAI
client = OpenAI()

Model = "gpt-4.1-nano"

### Message role 과 content
| 역할        | 설명                                                                 |
|-------------|----------------------------------------------------------------------|
| developer   | 애플리케이션 개발자가 제공한 지시사항이며, 사용자 메시지보다 우선시됩니다. |
| user        | 최종 사용자가 제공한 지시사항이며, developer 메시지보다 우선순위가 낮습니다. |
| assistant   | 모델이 생성한 메시지로, 응답의 역할을 합니다.                            |

In [14]:
response = client.chat.completions.create(
  model=Model,
  messages=[
    {"role": "developer", "content": "친절하고 예의바르게 말하세요."},
    {"role": "user", "content": "자바스크립트에서 세미콜론이 선택적인가요?"},
  ]
)

In [17]:
response.to_dict()

{'id': 'chatcmpl-Bdm2eAunufJqjhb7NwWPBP5hqc9vD',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '네, 자바스크립트에서는 세미콜론이 선택적입니다. 즉, 대부분의 경우에 세미콜론을 꼭 넣지 않아도 코드가 제대로 작동할 수 있습니다. 이를 "세미콜론 자동 삽입(Automatic Semicolon Insertion, ASI)"이라고 하는데요, 자바스크립트 엔진이 필요에 따라 세미콜론을 자동으로 추가해줍니다.\n\n그러나, 세미콜론을 생략할 때는 예상치 못한 동작이 발생할 수 있으니, 명시적으로 넣는 것이 더 안전하고 명확한 코드를 작성하는 방법입니다. 일관성을 위해 대부분의 개발자들은 세미콜론을 사용하는 것을 권장합니다.\n\n혹시 더 궁금하신 점이나, 구체적인 상황이 있다면 언제든 물어보세요!',
    'refusal': None,
    'role': 'assistant',
    'annotations': []}}],
 'created': 1748819164,
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_f12167b370',
 'usage': {'completion_tokens': 180,
  'prompt_tokens': 37,
  'total_tokens': 217,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 

In [18]:
print(response.choices[0].message.content)

네, 자바스크립트에서는 세미콜론이 선택적입니다. 즉, 대부분의 경우에 세미콜론을 꼭 넣지 않아도 코드가 제대로 작동할 수 있습니다. 이를 "세미콜론 자동 삽입(Automatic Semicolon Insertion, ASI)"이라고 하는데요, 자바스크립트 엔진이 필요에 따라 세미콜론을 자동으로 추가해줍니다.

그러나, 세미콜론을 생략할 때는 예상치 못한 동작이 발생할 수 있으니, 명시적으로 넣는 것이 더 안전하고 명확한 코드를 작성하는 방법입니다. 일관성을 위해 대부분의 개발자들은 세미콜론을 사용하는 것을 권장합니다.

혹시 더 궁금하신 점이나, 구체적인 상황이 있다면 언제든 물어보세요!


## 마크다운 및 XML을 사용한 메시지 서식 지정
- 개발자 및 사용자 메시지를 작성할 때 마크다운 서식과 XML 태그를 함께 사용하면 모델이 프롬프트 및 컨텍스트 데이터의 논리적 경계를 이해하도록 도울 수 있습니다.

- 마크다운 헤더와 목록은 프롬프트의 각 섹션을 구분하고 모델에 계층 구조를 전달하는 데 유용합니다. 또한 개발 과정에서 프롬프트의 가독성을 높일 수도 있습니다. XML 태그는 참조용으로 사용되는 지원 문서와 같은 콘텐츠의 시작과 끝을 구분하는 데 도움이 됩니다. XML 속성을 사용하여 지침에서 참조할 수 있는 프롬프트 콘텐츠에 대한 메타데이터를 정의할 수도 있습니다.

일반적으로 개발자 메시지에는 다음 섹션이 포함되며, 일반적으로 다음 순서대로 구성됩니다(최적의 콘텐츠 및 순서는 사용하는 모델에 따라 다를 수 있음).

- 정체성(Identity): 어시스턴트의 목적, 커뮤니케이션 스타일 및 상위 목표를 설명합니다.  
- 지침(Instructions): 원하는 응답을 생성하는 방법에 대한 지침을 모델에 제공합니다. 어떤 규칙을 따라야 할까요? 모델은 무엇을 해야 하고, 무엇을 하지 말아야 할까요? 이 섹션에는 모델이 사용자 지정 함수를 호출하는 방법 등 사용 사례와 관련된 여러 하위 섹션이 포함될 수 있습니다.  
- 예(Example): 가능한 입력 예시와 모델의 원하는 출력을 함께 제공하세요.  
- 맥락(Context): 모델이 응답을 생성하는 데 필요한 추가 정보(예: 학습 데이터 외부의 비공개/독점 데이터 또는 특히 관련성이 있다고 생각되는 기타 데이터)를 제공하세요. 이러한 콘텐츠는 일반적으로 프롬프트 끝부분에 배치하는 것이 가장 좋습니다. 생성 요청에 따라 다른 맥락을 포함할 수 있기 때문입니다.

```
# Identity

JavaScript 코드에서 스네이크 케이스(snake case) 변수 사용을 강제하고 Internet Explorer 버전 6에서 실행되는 코드를 작성하는 데 도움이 되는 코딩 도우미입니다.

# Instructions

* 변수를 정의할 때는 카멜 케이스(camel case) 이름(예: myVariable) 대신 스네이크 케이스(snake case) 이름(예: my_variable)을 사용하세요.
* 이전 브라우저를 지원하려면 이전 "var" 키워드를 사용하여 변수를 선언하세요.
* 마크다운 형식으로 응답하지 말고, 요청된 코드를 그대로 반환하세요.

# Examples

<user_query>
이름에 대한 문자열 변수를 선언하려면 어떻게 해야 하나요?
</user_query>

<assistant_response>
var first_name = "Anna";
</assistant_response>
```

In [31]:
with open("prompt.txt", "r", encoding="utf-8") as f:
    instructions = f.read()

response = client.responses.create(
    model=Model,
    instructions=instructions,
    input="이름의 성을 나타내는 변수를 어떻게 선언하나요?",
)

print(response.output_text)

var last_name = "";


In [33]:
response.to_dict()

{'id': 'resp_683ce4cab1848195aeb3b37377e3d88208f0722ed93b3a95',
 'created_at': 1748821194.0,
 'error': None,
 'incomplete_details': None,
 'instructions': '# Identity\n\nYou are coding assistant that helps enforce the use of snake case \nvariables in JavaScript code, and writing code that will run in \nInternet Explorer version 6.\n\n# Instructions\n\n* When defining variables, use snake case names (e.g. my_variable) \n  instead of camel case names (e.g. myVariable).\n* To support old browsers, declare variables using the older \n  "var" keyword.\n* Do not give responses with Markdown formatting, just return \n  the code as requested.\n\n# Examples\n\n<user_query>\nHow do I declare a string variable for a first name?\n</user_query>\n\n<assistant_response>\nvar first_name = "Anna";\n</assistant_response>',
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [{'id': 'msg_683ce4cae21c819583f134c3b66d188308f0722ed93b3a95',
   'content': [{'annotations': 

## 프롬프트 캐싱으로 비용 및 지연 시간 절약
메시지를 작성할 때 API 요청에서 반복적으로 사용될 것으로 예상되는 콘텐츠는 프롬프트 시작 부분과 채팅 완료 또는 응답에 JSON 요청 본문으로 전달하는 첫 번째 API 매개변수에 포함해야 합니다. 이렇게 하면 프롬프트 캐싱을 통해 비용 및 지연 시간을 최대한 절약할 수 있습니다.  
따라서, 자주 쓰는 내용(예: system 프롬프트, 공통 프레임워크)은 프롬프트의 앞부분에 위치시키라는 것이 OpenAI 의 권고 사항 입니다.

# 퓨샷 러닝(Few-shot learning)
퓨샷 러닝을 사용하면 모델을 미세 조정하는 대신 프롬프트에 소수의 입력/출력 예제를 포함하여 대규모 언어 모델을 새로운 작업으로 유도할 수 있습니다. 모델은 이러한 예제에서 패턴을 암묵적으로 "취득"하여 프롬프트에 적용합니다. 예제를 제공할 때는 원하는 출력을 가진 다양한 입력을 하세요.

일반적으로 API 요청 시 개발자 메시지의 일부로 예제를 제공합니다. 다음은 모델이 긍정적 또는 부정적 고객 서비스 리뷰를 분류하는 방법을 보여주는 예제가 포함된 개발자 메시지 예시입니다.

```
# Identity

당신은 짧은 제품 리뷰를 긍정적, 부정적, 중립적으로 분류하는 유용한 도우미입니다.

# Instructions

* 답변에는 추가 서식이나 설명 없이 한 단어만 입력하세요.

* 제품 리뷰의 감정에 따라 "긍정적", "부정적" 또는 "중립적" 중 하나만 입력해야 합니다.

# Examples

<product_review id="example-1">
이 헤드폰 정말 좋아요. 음질이 정말 훌륭해요!
</product_review>

<assistant_response id="example-1">
긍정적
</assistant_response>

<product_review id="example-2">
배터리 수명은 괜찮은데 이어패드가 싸구려 같아요.
</product_review>

<assistant_response id="example-2">
중립
</assistant_response>

<product_review id="example-3">
고객 서비스가 최악이에요. 다시는 이 제품 안 살 거예요.
</product_review>

<assistant_response id="example-3">
부정적
</assistant_response>
```

In [22]:
completion = client.chat.completions.create(
    model=Model,
    messages=[
        {"role": "system", "content": "당신은 짧은 제품 리뷰를 긍정적, 부정적, 중립적으로 분류하는 유용한 도우미입니다.\n\n* 답변에는 추가 서식이나 설명 없이 한 단어만 입력하세요.\n* 제품 리뷰의 감정에 따라 \"긍정적\", \"부정적\" 또는 \"중립적\" 중 하나만 입력해야 합니다."},
        
        # Few-shot examples
        {"role": "user", "content": "이 헤드폰 정말 좋아요. 음질이 정말 훌륭해요!"},
        {"role": "assistant", "content": "긍정적"},
        
        {"role": "user", "content": "배터리 수명은 괜찮은데 이어패드가 싸구려 같아요."},
        {"role": "assistant", "content": "중립"},
        
        {"role": "user", "content": "고객 서비스가 최악이에요. 다시는 이 제품 안 살 거예요."},
        {"role": "assistant", "content": "부정적"},

        # Your new review to classify
        {"role": "user", "content": "배송은 빨랐지만 제품이 고장 났어요."}
    ]
)

print(completion.choices[0].message.content)

부정적


In [24]:
with open("fewshot_example.txt", "r", encoding="utf-8") as f:
    instructions = f.read()

response = client.responses.create(
    model=Model,
    instructions=instructions,
    input="배송은 빨랐지만 제품이 고장 났어요.",
)

print(response.output_text)

부정적


### RAG(Retrieval Augmented Generation)을 이용한 질문 답변

In [35]:
import requests
from bs4 import BeautifulSoup

# Wikipedia 페이지의 URL
#url = "https://en.wikipedia.org/wiki/Curling_at_the_2022_Winter_Olympics"
url = "https://namu.wiki/w/2022%20%EB%B2%A0%EC%9D%B4%EC%A7%95%20%EB%8F%99%EA%B3%84%EC%98%AC%EB%A6%BC%ED%94%BD/%EC%BB%AC%EB%A7%81"

# 페이지 내용 가져오기
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# 페이지 내용 중에서 필요한 부분 추출 (예시로 본문 전체를 가져옵니다)
wikipedia_article_on_curling = soup.get_text()
wikipedia_article_on_curling

'\n\n\n2022 베이징 동계올림픽/컬링 - 나무위키\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n최근 변경최근 토론특수 기능 \xa02022 베이징 동계올림픽/컬링 최근 수정 시각: 2025-04-15 06:39:394편집토론역사분류2022 베이징 동계올림픽동계올림픽/컬링2022베이징 동계올림픽 [ 펼치기 · 접기 ]주요 정보유치 과정 · 경기 종목 · 선수 목록 · 대회 진행 (개막식 · 폐막식) · 대한민국 대표팀 · 평가  문제점 및 사건 사고 (대규모 외교적 보이콧 · 문화공정 논란ㆍ쇼트트랙 편파판정 논란 (반응) · 카밀라 발리예바 도핑 적발 사건) 개별 종목 쇼트트랙 스피드 스케이팅 ·  피겨 스케이팅 ·  컬링 ·  루지 ·  아이스하키대회 관련 기타 문서마스코트 빙둔둔 · 주경기장 베이징국가체육장 · 패럴림픽 2022 베이징 동계패럴림픽 · 개막 선언자 시진핑 · 성화 점화자 디니걸 이라무장 · 자오자원一起向未来Together For a Shared Future함께하는 미래로2022베이징 동계패럴림픽 [ 펼치기 · 접기 ]주요 정보유치 과정 · 대한민국 대표팀대회 관련 기타 문서마스코트 쉐룽룽 · 주경기장 베이징국가체육장 · 올림픽 2022 베이징 동계올림픽 · 개막 선언자 시진핑 · 성화 점화자 리두안一起向未来Together For a Shared Future함께하는 미래로2022 베이징 동계올림픽 컬링날짜2022년 2월 2일 ~ 2월 20일경기장국립 아쿠아틱 센터종목믹스더블, 남자, 여자챔피언믹스더블 이탈리아남자 스웨덴여자 영국← 20182026 →\xa01. 소개2. 경기 일정3. 출전권3.1. 상세4. 참가 선수 명단4.1. 남자4.1.1. 각 팀 스킵 간략 정보4.2. 여자4.2.1. 각 팀 스킵 간략 정보4.3. 믹스더블4.3.1. 각 팀 간략 정보5. 경기 결과5.1. 믹스더블5.1.1. 라운드 로빈5.1.2. 결선 토너먼트5.2. 남자5.2.1. 라운드 로빈5.2.2. 결선 토너먼트5.3

In [39]:
query = f"""다음 질문에 답하려면 2022년 동계 올림픽에 관한 아래 기사를 사용하세요. 답을 찾을 수 없으면 "모르겠어요."라고 답하세요.

Article:
\"\"\"
{wikipedia_article_on_curling}
\"\"\"

질문: 2022년 동계 올림픽 컬링 금메달을 획득한 선수는 누구인가요?"""

response = client.chat.completions.create(
    messages=[
        {'role': 'developer', 'content': '2022년 동계올림픽에 관한 질문에 답변합니다..'},
        {'role': 'user', 'content': query},
    ],
    model=Model,
    temperature=0,
)

print(response.choices[0].message.content)

2022년 동계 올림픽 컬링 여자부에서 금메달을 획득한 선수는 일본 팀의 멤버들입니다.


### 모든 응답에는 'finish_reason'이 포함되며, 'finish_reason'의 가능한 값은 다음과 같습니다:
- stop: API가 완전한 메시지를 반환하거나, stop 매개변수를 통해 제공된 중단 시퀀스 중 하나에 의해 메시지가 종료됨
- length: max_tokens 매개변수 또는 토큰 제한으로 인해 완전하지 않은 모델 출력
- function_call: 모델이 함수를 호출하기로 결정함

In [40]:
print(response.choices[0].finish_reason)

stop


## Token 관리
- tiktoken은 OpenAI 모델과 함께 사용할 수 있는 빠른 BPE(https://en.wikipedia.org/wiki/Byte-pair_encoding) 토크나이저입니다.
- "ChatGPT is great!“ 는 6개의 토큰으로 인코딩됩니다 $\rightarrow$ ["Chat", "G", "PT", " is", " great", "!"]
- 예를 들어, API 호출이 메시지 입력에서 10개의 토큰을 사용하고 메시지 출력에서 ​​20개의 토큰을 받은 경우 30개의 토큰에 대한 요금이 청구됩니다.


In [42]:
print(response.usage)
print(response.usage.prompt_tokens)

CompletionUsage(completion_tokens=32, prompt_tokens=9056, total_tokens=9088, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
9056


In [43]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

In [47]:
messages=[
    {"role": "developer", "content": "You are a helpful assistant designed to output JSON."},
    {"role": "user", "content": "2020년 코리안 시리즈를 우승한 야구팀이 어디?"},
  ]

num_tokens = 0
for message in messages:
  num_tokens += 4  # every message follows <im_start>{role/name}\n{content}<im_end>\n
  for key, value in message.items():
      num_tokens += len(enc.encode(value))
      print(value)
      print(enc.encode(value))

num_tokens

developer
[77944]
You are a helpful assistant designed to output JSON.
[3575, 553, 261, 10297, 29186, 6884, 316, 4733, 8205, 13]
user
[1428]
2020년 코리안 시리즈를 우승한 야구팀이 어디?
[1323, 15, 12622, 43824, 4960, 14307, 12468, 186042, 4831, 26090, 42643, 3748, 109887, 10997, 91214, 2186, 154074, 30]


38

### 모델에 인터넷의 최신 정보에 대한 액세스 권한을 부여
- `gpt-4o-search-preview` 또는 `gpt-4o-mini-search-preview` 모델 사용

In [49]:
completion = client.chat.completions.create(
    model="gpt-4o-mini-search-preview",
    web_search_options={},
    messages=[
        {
            "role": "user",
            "content": "최근에 한국에서 나온 큰 뉴스가 뭐야?",
        }
    ],
)

print(completion.choices[0].message.content)

최근 한국에서 주목받은 주요 뉴스는 다음과 같습니다:

1. **영화 '히든 페이스' 흥행**: 김대우 감독의 신작 영화 '히든 페이스'가 개봉 3주 만에 관객 수 90만 명을 돌파하며, 최근 5년간 '청소년 관람불가' 등급을 받은 한국 영화 중 최고 성적을 기록했습니다. ([news.nate.com](https://news.nate.com/view/20241210n14211?utm_source=openai))

2. **임영웅의 트로트 인기 상승**: 가수 임영웅이 방탄소년단(BTS)을 제치고 한국에서 가장 사랑받는 가수로 선정되었으며, 그의 콘서트 투어 매출은 302억 원을 돌파하여 그룹이 아닌 아티스트 중 최고 기록을 세웠습니다. ([fnnews.com](https://www.fnnews.com/news/202408061715293530?utm_source=openai))

3. **K-푸드 수출 증가**: 김과 즉석밥 등 한국의 가공식품 수출이 역대 최대치를 기록했습니다. 특히, 미국에서 냉동 김밥의 인기가 높아지며 가공밥 수출액은 올해 10월까지 7,900만 달러를 달성했습니다. ([imnews.imbc.com](https://imnews.imbc.com/replay/2023/nwdesk/article/6548466_36199.html?utm_source=openai))

4. **제주도 쓰레기 문제**: 제주도에서 관광객 증가로 인해 쓰레기 수거가 원활하지 않아 거리마다 쓰레기가 넘치는 상황이 우려되고 있습니다. ([balpumnews.com](https://www.balpumnews.com/news/jeju-island-trash-chaos-2024?utm_source=openai))

5. **'닭강정' 논란**: 넷플릭스 오리지널 시리즈 '닭강정'이 사우디아라비아 왕실을 언급한 장면으로 인해 아랍권 시청자들로부터 비판을 받았습니다. 제작진은 해당 장면이 허구의 이야기일 뿐이라고 해명했습니다. ([maxmovieen.com](https://www

## 실습 : 위의 messages 내용을 각자 수정해 가면서 API 실행